# SOXX/SOXL 2021 指数变更原始来源
只验证 SEC 原文中的 PHLX→ICE 结构变化，不读取市场结果。

In [1]:
import hashlib,json,shlex,subprocess,tarfile,tempfile
from pathlib import Path
from bs4 import BeautifulSoup
from pypdf import PdfReader
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd().resolve();card=root/'research/stage1_semiconductor_index_regime_source_20261023';plan=json.loads((card/'plan.json').read_text());receipt=json.loads((card/'capture_receipts.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and receipt['plan_sha256']==sha(card/'plan.json') and receipt['GETs_attempted']==len(plan['exact_requests'])==3
texts=[];paths=[card/'plan.json',card/'capture_receipts.json']
for i,(case,row) in enumerate(zip(plan['exact_requests'],receipt['captures'],strict=True),1):
 assert row['role']==case['role'] and row['requested_url']==case['url'] and row['curl_exit']==0 and row['http_status']=='200'
 for kind in ('body','headers'):
  info=row[kind];path=root/info['path'];assert path.stat().st_size==info['bytes'] and sha(path)==info['sha256'];paths.append(path)
 body=root/row['body']['path']
 if i<3:text=BeautifulSoup(body.read_bytes(),'html.parser').get_text(' ',strip=True)
 else:
  assert body.read_bytes()[:4]==b'%PDF';text=' '.join(x.extract_text() or '' for x in PdfReader(body).pages)
 normalized=' '.join(text.replace('\xa0',' ').split());texts.append(normalized)
 assert all(term.lower() in normalized.lower() for term in case['required']),(case['role'],case['required'])
assert 'PHLX Semiconductor Sector Index' in texts[0] and 'PHLX Semiconductor Sector Index' in texts[1]
assert 'on or around June 21, 2021' in texts[0] and 'On or about August 25, 2021' in texts[1]
assert 'August 27, 2021' in texts[2] and 'ICE Semiconductor Index' in texts[2] and 'SOXL' in texts[2]
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='semi-index-regime-source-') as directory:
 archive=Path(directory)/'raw.tar.gz'
 with tarfile.open(archive,'w:gz') as output:
  for path in paths:output.add(path,arcname=str(path.relative_to(root)),recursive=False)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 restored=Path(directory)/'restored.tar.gz'
 with restored.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,timeout=60)
 assert transfer.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as source:
  members={m.name:m for m in source.getmembers()};assert set(members)=={str(p.relative_to(root)) for p in paths}
  for path in paths:assert source.extractfile(members[str(path.relative_to(root))]).read()==path.read_bytes()
 evidence['nas_path']=remote_path
checks=[{'role':case['role'],'body_sha256':row['body']['sha256'],'response_bytes':row['body']['bytes'],'required_terms_present':True} for case,row in zip(plan['exact_requests'],receipt['captures'],strict=True)]
result={'status':'official_2021_SOXX_SOXL_common_index_transition_source_passed_conservative_boundary','plan_sha256':sha(card/'plan.json'),'GETs_consumed':3,'checks':checks,'source_interpretation':{'SOXX':'2021 SEC supplement prospectively says on or around June 21 PHLX-to-ICE change','SOXL':'2021 SEC supplement prospectively says on or about August 25 PHLX-to-ICE change','SOXL_Aug27_listing':'SEC listing report dated August 27 names ICE Semiconductor Index','conservative_regime_dates':plan['conservative_regime_dates'],'exact_SOXX_effective_date_certified_by_these_three_bytes':False},'NAS_evidence':evidence,'NAS_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':'conditional_official_2021_index_transition_boundary_source_passed_no_outcomes','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','GETs_consumed':3,'NAS_archive_sha256':evidence['sha256'],'NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],evidence['sha256'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


conditional_official_2021_index_transition_boundary_source_passed_no_outcomes a3f14d3f7d89bbef40cb01aa8b381afaae5f0e4e86e473f48a624d05b14fbdb4


## 中文结论
# SOXX 与 SOXL 的 2021 年底层指数变更

冻结三份官方 SEC 原件各请求一次，HTTP 均为 200；原文、响应头、URL、UTC 请求时刻与 SHA-256 已保存在本机并归档 Yanxuan NAS，远端哈希和限定逐文件恢复通过。响应头含临时 Cookie，原字节不提交 Git。0 市场收益读取、0 拟合、0 账户。

[SOXX 的 SEC 补充文件](https://www.sec.gov/Archives/edgar/data/1100663/000119312521126827/d82642d497.htm)当时预告在 2021-06-21 **左右**从 PHLX Semiconductor Sector Index 改为 ICE Semiconductor Index。[SOXL 的 SEC 补充文件](https://www.sec.gov/Archives/edgar/data/1424958/000119312521195149/d177232d497.htm)预告在 2021-08-25 **左右**作相同更换；[8 月 27 日交易所申报](https://www.sec.gov/Archives/edgar/vprr/2100/21009136.pdf)已将 ICE 列为 SOXL 底层指数。由此可将 2021 年夏季作为外生训练域变化的来源假设，但这三份原文没有认证 SOXX 的确切生效秒点。后续只读归因应把 2021-06-14 至 08-27 保留为过渡未知，2021-06-11 及以前作旧域，2021-08-30 起作共同新域，避免用“左右”硬切交易日。

这个来源只证明两个基金发生过结构变化及保守时段；它不能说明 Stage1 失败由此造成，更不能把变更日期变成择时阈值。决定：`conditional_official_2021_index_transition_boundary_source_passed_no_outcomes`。下一步须在匹配 Qlib 原训练与复用评价价格面上，比较旧域、过渡、新域的 SOXL−SOXX 跟踪残差及标签分布；再判断是否值得冻结一个真正的 Stage1 0bp 新机制。SOXX 目标尚未实现。
